In [5]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from artifact_loader import original_data_path

In [6]:
Path.cwd().name == "notebooks"

True

In [7]:
df_cbarq = pd.read_csv(
    '../datatsets/foreign/bohland_2023_shelter_behavior_after_adoption_supplement_s4.csv'
)

In [8]:
import re

def canonical_name(column: str) -> str:
    return re.sub(r"\.\d+$", "", column)

column_groups = {}
for column in df_cbarq.columns:
    column_groups.setdefault(canonical_name(column), []).append(column)

four_wave_fields = {
    name: [name, f"{name}.1", f"{name}.2", f"{name}.3"]
    for name, columns in column_groups.items()
    if set(columns) == {name, f"{name}.1", f"{name}.2", f"{name}.3"}
}

len(four_wave_fields)

80

In [9]:
# Валидированный набор полей, повторяющихся во всех четырёх замерах.
metadata_field_names = {
    "Date/Time Completed:",
    "Dog shelter name: ",
}

missing_metadata = metadata_field_names.difference(four_wave_fields)
assert not missing_metadata, f"Metadata fields not found: {missing_metadata}"

questionnaire_fields = {
    name: columns
    for name, columns in four_wave_fields.items()
    if name not in metadata_field_names
}

metadata_fields = {
    name: four_wave_fields[name]
    for name in metadata_field_names
}

assert len(four_wave_fields) == 80
assert len(questionnaire_fields) == 78
assert len(metadata_fields) == 2
assert set(questionnaire_fields).isdisjoint(metadata_fields)

field_catalog = pd.DataFrame(
    [
        {
            "canonical_name": name,
            "field_type": "metadata" if name in metadata_fields else "questionnaire",
            "wave_1_column": columns[0],
            "wave_2_column": columns[1],
            "wave_3_column": columns[2],
            "wave_4_column": columns[3],
        }
        for name, columns in four_wave_fields.items()
    ]
)

field_catalog["field_type"].value_counts()

field_type
questionnaire    78
metadata          2
Name: count, dtype: int64

In [10]:
# Формируем отдельный DataFrame для каждой волны с единой схемой колонок.
wave_config = {
    1: {"event_name": "10 days after adoption", "wave_days": 10, "column_index": 0},
    2: {"event_name": "30 days after adoption", "wave_days": 30, "column_index": 1},
    3: {"event_name": "3 months after adoption", "wave_days": 90, "column_index": 2},
    4: {"event_name": "6 months after adoption", "wave_days": 180, "column_index": 3},
}

canonical_field_order = list(four_wave_fields)
wave_observations = {}

for observation_number, config in wave_config.items():
    source_columns = [
        four_wave_fields[name][config["column_index"]]
        for name in canonical_field_order
    ]
    rename_map = dict(zip(source_columns, canonical_field_order))

    wave_df = (
        df_cbarq.loc[df_cbarq["Event Name"].eq(config["event_name"])]
        .reindex(columns=["Record ID", "Event Name", *source_columns])
        .rename(columns=rename_map)
        .copy()
    )
    wave_df["observation_number"] = observation_number
    wave_df["wave_days"] = config["wave_days"]
    wave_df = wave_df[
        [
            "Record ID",
            "observation_number",
            "Event Name",
            "wave_days",
            *canonical_field_order,
        ]
    ]

    assert wave_df["Event Name"].eq(config["event_name"]).all()
    assert wave_df.columns.is_unique
    wave_observations[observation_number] = wave_df

pd.DataFrame(
    [
        {
            "observation_number": number,
            "event_name": wave_config[number]["event_name"],
            "wave_days": wave_config[number]["wave_days"],
            "rows": len(wave_df),
            "columns": wave_df.shape[1],
        }
        for number, wave_df in wave_observations.items()
    ]
)

,observation_number,event_name,wave_days,rows,columns
0,1,10 days after adoption,10,99,84
1,2,30 days after adoption,30,83,84
2,3,3 months after adoption,90,75,84
3,4,6 months after adoption,180,82,84


In [11]:
# Объединяем четыре замера вертикально: одна строка = одна собака в один момент времени.
df_cbarq_panel = pd.concat(
    [wave_observations[number] for number in sorted(wave_observations)],
    ignore_index=True,
)

assert df_cbarq_panel.shape == (339, 84)
assert not df_cbarq_panel.duplicated(["Record ID", "observation_number"]).any()

df_cbarq_panel.groupby(
    ["observation_number", "Event Name", "wave_days"],
    observed=True,
).size().rename("rows").reset_index()

,observation_number,Event Name,wave_days,rows
0,1,10 days after adoption,10,99
1,2,30 days after adoption,30,83
2,3,3 months after adoption,90,75
3,4,6 months after adoption,180,82
